In [1]:
import os
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import time
import datetime
from sqlalchemy import create_engine
import pymysql
pymysql.install_as_MySQLdb()

In [143]:
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = Options()
options.add_experimental_option("detach", True)
options.add_argument("start-maximized")
options.add_argument("Chrome/135.0.0.0")
options.add_argument("lang=ko_KR")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
    )

## url 가져오기
url="https://play.google.com/store/apps/details?id=viva.republica.toss"
driver.get(url)

In [144]:
# 평점 및 리뷰 팝업창 열기
review_button =driver.find_element(By.CSS_SELECTOR,'button[aria-label="평점 및 리뷰 자세히 알아보기"]')
review_button.click()

In [19]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [145]:
# 팝업 창 정보 가져오기
wait = WebDriverWait(driver, 10)
review_page=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,"div.fysCi.Vk3ZVd")))
#review_page

In [50]:
# 최신순으로 리뷰 정렬
# 정렬 버튼
order_button=driver.find_element(By.CSS_SELECTOR,'div[aria-label="관련성순"]')
order_button.click()

#최신순 선택
order_new=driver.find_element(By.CSS_SELECTOR,".z80M1.NmX0eb.KnEF3e[aria-label='최신']")
order_new.send_keys(Keys.ENTER)

In [146]:
max_retries=2
retries = 0
while retries < max_retries:
    try:
        # Find the "관련성순" button
        order_button = driver.find_element(By.CSS_SELECTOR, 'div[aria-label="관련성순"]')
        order_button.click()

        # Select the "최신" option
        order_new = driver.find_element(By.CSS_SELECTOR, ".z80M1.NmX0eb.KnEF3e[aria-label='최신']")
        order_new.send_keys(Keys.ENTER)
        break  # Exit the loop if the operation is successful
    except (NoSuchElementException, ElementNotInteractableException) as e:
        retries += 1
        if retries >= max_retries:
            raise Exception(f"Failed to sort reviews by newest after {max_retries} retries. Error: {e}")
        print(f"Failed to sort reviews by newest. Retrying ({retries}/{max_retries})...")

In [109]:
import re
#리뷰 리스트
review_list=driver.find_elements(By.CSS_SELECTOR,".RHo1pe")
for review in review_list:
    #작성자 닉네임
    print(review.find_element(By.CSS_SELECTOR,".X5PpBb").text)
    #별점
    aria_label=review.find_element(By.CSS_SELECTOR,'div[aria-label^="별표 5개 만점에 "]').get_attribute('aria-label')
    rating_pattern=r'별표 5개 만점에 (\d)개를 받았습니다\.'
    rating = re.search(rating_pattern, aria_label).group(1)
    print(rating)
    #작성 날짜
    print(review.find_element(By.CSS_SELECTOR,".bp9Aid").text)
    # 리뷰 내용
    print(review.find_element(By.CSS_SELECTOR,".h3YV2d").text)
    # 개발자 답변
    try:
        print(review.find_element(By.CSS_SELECTOR,".ras4vb").text)
    except Exception:
        print('답변 없음')
    # 공감 수
    try:
        useful_label=review.find_element(By.CSS_SELECTOR,".AJTPZc").text
        useful_pattern=r'사용자 (\d+)명이 이 리뷰가 유용하다고 평가함'
        useful = re.search(useful_pattern, useful_label).group(1)
        print(useful)
    except Exception:
        thumbs_up=None

정보영
1
2025년 4월 10일
자주쓰는계좌 업데이트 되었던데 이게 뭔가요.... 전처럼 사용자가 알아서 설정하게 원상복구 시켜주세요
안녕하세요. 정보영 님, 토스팀입니다. 서비스 변경으로 인해 불편을 드린 것 같아 마음이 무겁습니다. 말씀주신 부분은 현재 점진적으로 모든 고객님들 대상으로 변경이 될 예정으로 이전 홈 화면으로의 변경은 어려운 점 너그러운 마음으로 양해 부탁드립니다. 귀한 시간 내어 전달해주신만큼 담당부서로 전달 될 수 있도록 하겠으며, 더 나은 서비스를 제공 위해 노력하겠습니다. 보다 자세한 확인 및 안내를 위해 24시간 운영되는 카카오톡(@toss) 또는 고객센터 1599-4905 로 편하실 때 문의 부탁 드려도 될까요? 일부 안내의 경우 리뷰를 통한 즉각적인 안내가 어려울 수 있는 점 양해 부탁 드립니다. 감사합니다.
내가왕이다
3
2025년 4월 10일
요 며칠 토스 친구가 안뜨네요. 저만 안뜨는게 아니고 주변에 몇 몇도 안뜬대요. 뜨는 사람들은 제 포인트를 먹는데 같이 블루투스 켜고 접속해도 저만 안뜰 많네요..
안녕하세요. 내가왕이다 님, 토스팀입니다. 1일 이내 상대를 만나 클릭한 기록이 있다면 다음날 00시까지 표시되지 않습니다. 위에 해당하지 않는다면 토스 앱 버전이 최신 버전인지 확인 하시어, ⓐ 기기 재부팅 후 블루투스 on/off 하여 재시도, ⓑ 기기 내의 블루투스 설정하는 화면에서 상대 기기가 정상적으로 검색 되는지, ⓒ 휴대폰 설정 - 애플리케이션 - 토스 - 위치권한이 활성화 되어 있는지, ⓓ 페어링 될 기기가 주변에 너무 많거나, 가로 막혀진 상황은 아닌지 상대방 기기에도 동일한 환경이어야 하므로 함께 확인해보실 수 있을까요? 그럼에도 동일하다면 24시간 운영되는 토스 고객센터로 문의 부탁드립니다.
유승민
1
2025년 4월 10일
카드 충전이 안되는데 무슨
안녕하세요. 유승민 님, 토스팀입니다. 남겨주신 내용만으로는 정확한 확인이 어려운 점 양해 말씀 드립니다. 보다 자세한 확인 및 안내를 위해 24시간 운영되

만보기 복권 긁고나면 적립이 안되거나 광고창에서 카운트가 멈추는 현상이 지속되고 있습니다. 업데이트도 해보고 새로깔아봐도 동일 증상 지속되고 있네요;;
안녕하세요. 안창인 님, 토스팀입니다. 토스 이용에 불편을 드려 죄송합니다. 다만 남겨주신 내용만으로는 정확한 확인이 어려운 점 양해 말씀 드립니다. 보다 자세한 확인 및 안내를 위해 24시간 운영되는 카카오톡(@toss) 또는 고객센터 1599-4905 로 편하실 때 문의 부탁 드려도 될까요? 일부 안내의 경우 리뷰를 통한 즉각적인 안내가 어려울 수 있는 점 양해 부탁 드립니다. 감사합니다.
1
손주영
1
2025년 4월 8일
만보기 복권 실수로 광고 다 보고 뒤로가기 눌렀더니 그 뒤로 복권 긁고나면 흰 화면밖에 안보이고 돈을 못받아요..
안녕하세요. 손주영 님, 토스팀입니다. 토스 이용에 불편을 드려 죄송합니다. 다만 남겨주신 내용만으로는 정확한 확인이 어려운 점 양해 말씀 드립니다. 보다 자세한 확인 및 안내를 위해 24시간 운영되는 카카오톡(@toss) 또는 고객센터 1599-4905 로 편하실 때 문의 부탁 드려도 될까요? 일부 안내의 경우 리뷰를 통한 즉각적인 안내가 어려울 수 있는 점 양해 부탁 드립니다. 감사합니다.
2
이강원
1
2025년 4월 8일
갤럭시 s25 울트라 차트볼때마다 앱크래시 겁나많이나네요 제발 수정좀
안녕하세요. 이강원 님, 토스팀입니다. 우선 토스증권 이용에 불편을 드려 죄송하다는 말씀 드립니다. 고객님께서 말씀해주신 내용은 팀내에 공유하여 보다 편리한 토스증권 사용 경험을 하실 수 있도록 노력 하겠습니다. 다른 문의사항이 있으시다면 보다 자세한 확인 및 안내를 위해 토스증권 고객센터 1599-7987 로 문의 부탁 드려도 될까요? 일부 안내의 경우 리뷰를 통한 즉각적인 안내가 어려울 수 있는 점 양해 부탁 드립니다. 감사합니다.
1
한기성
5
2025년 4월 8일
토스 카드 예쁜다!
답변 없음
Wjoo
1
2025년 4월 8일
오늘 토스로 대출받있는데 카뱅 대출에 비해 높은

In [150]:
# 리뷰가 담긴 창을 찾아서 JavaScript로 8000px씩 아래로 스크롤
driver.execute_script("document.querySelector('.fysCi.Vk3ZVd').scrollBy(0, 8000)")

In [164]:

import re
#리뷰 리스트
review_list=driver.find_elements(By.CSS_SELECTOR,".RHo1pe")
for review in review_list:
    #작성자 닉네임
    nickname=review.find_element(By.CSS_SELECTOR,".X5PpBb").text
    #별점
    aria_label=review.find_element(By.CSS_SELECTOR,'div[aria-label^="별표 5개 만점에 "]').get_attribute('aria-label')
    rating_pattern=r'별표 5개 만점에 (\d)개를 받았습니다\.'
    rating = re.search(rating_pattern, aria_label).group(1)
    #작성 날짜
    date_label=review.find_element(By.CSS_SELECTOR,".bp9Aid").text
    date_pattern=r'(\d+)년 (\d+)월 (\d+)일'
    review_date=re.search(date_pattern, date_label)
    review_date=f"{int(review_date.group(1))}.{int(review_date.group(2)):02d}.{int(review_date.group(3)):02d}"
    # 리뷰 내용
    review_content=review.find_element(By.CSS_SELECTOR,".h3YV2d").text
    # 개발자 답변
    try:
        answer=review.find_element(By.CSS_SELECTOR,".ras4vb").text
    except Exception:
        answer=None
    # 공감 수
    try:
        useful_label=review.find_element(By.CSS_SELECTOR,".AJTPZc").text
        useful_pattern=r'사용자 (\d+)명이 이 리뷰가 유용하다고 평가함'
        useful = int(re.search(useful_pattern, useful_label).group(1))
    except Exception:
        useful=0
    temp=(nickname, rating, review_date, review_content, answer, useful)
    columns=('닉네임','별점','작성 날짜','리뷰','개발자 답변','공감 수')
    result=pd.DataFrame([temp],columns=columns)
    display(result)

,닉네임,별점,작성 날짜,리뷰,개발자 답변,공감 수
0,Yoseop Jeong,1,2025.04.10,메인화면에 노출되는 계좌 원래대로 다 나오게 돌려주세요 아니면 최소 사용자가 지정할...,"안녕하세요. Yoseop Jeong 님, 토스팀입니다. 서비스 변경으로 인해 불편을...",0


,닉네임,별점,작성 날짜,리뷰,개발자 답변,공감 수
0,바다김,3,2025.04.10,업뎃후 알림 바로가기안됨,"안녕하세요. 바다김 님, 토스팀입니다. 남겨주신 내용만으로는 정확한 확인이 어려운 ...",4


,닉네임,별점,작성 날짜,리뷰,개발자 답변,공감 수
0,Itne Upne,1,2025.04.10,두더지 잡기 부정행위 했다는데 나는 아무짓도 안했음,"안녕하세요, 고객님. 토스뱅크 고객센터 입니다. 두더지게임은 여러 고객님에게 혜택과...",0


,닉네임,별점,작성 날짜,리뷰,개발자 답변,공감 수
0,박태환,1,2025.04.10,아니 접근성이 꺼져있다면서 나가게 하면서 기회도 다시안주는 토스 대단하다,"안녕하세요. 박태환 님, 토스팀입니다. 남겨주신 내용만으로는 정확한 확인이 어려운 ...",0


,닉네임,별점,작성 날짜,리뷰,개발자 답변,공감 수
0,INY L,1,2025.04.10,너무 느려요. 로딩이 더딜때가 많아요. 다른앱들도 그러면 내핸드폰이 이상한가 할텐데...,"안녕하세요. INY L 님, 토스팀입니다. 우선 원활한 이용에 불편을 드려 죄송합니...",0


...

,닉네임,별점,작성 날짜,리뷰,개발자 답변,공감 수
0,ByoungGeol Cho,1,2025.04.03,김밥 자르기 5천원까지 준다고 하더니 50원 넘어가니깐 0원만 나오네 사기 아님?,"안녕하세요. ByoungGeol Cho 님, 토스뱅크입니다. 남겨주신 내용만으로는 ...",0


,닉네임,별점,작성 날짜,리뷰,개발자 답변,공감 수
0,박상주,1,2025.04.03,첨엔 그러려니 했지만 갈수록 해도해도 광고가 너무 심합니다,"안녕하세요. 박상주 님, 토스팀입니다. 먼저 서비스 이용에 불편을 드렸다면 대단히 ...",0


,닉네임,별점,작성 날짜,리뷰,개발자 답변,공감 수
0,조주현,1,2025.04.03,환금액 8만얼마 나온다더니 갑자기 찾을 환금액 없데 니들이 먹냐?,"안녕하세요. 조주현 님, 토스팀입니다.남겨주신 내용만으로는 정확한 확인이 어려운 점...",0


,닉네임,별점,작성 날짜,리뷰,개발자 답변,공감 수
0,Andrew,3,2025.04.03,"메인 증권앱을 타 증권앱에서 토스앱으로 갈아타기 하였습니다만, 아래 내용이 개선되면...","안녕하세요. Andrew 님, 토스팀입니다. 먼저 토스증권 이용하시면서 느낀 소중한...",0


,닉네임,별점,작성 날짜,리뷰,개발자 답변,공감 수
0,최현준,1,2025.04.03,배당금 너무 늦게 주는거아님?,"안녕하세요. 최현준 님, 토스팀입니다. 우선 토스증권 서비스 이용에 불편을 드려 대...",0


In [163]:
#작성 날짜
review_date=review.find_element(By.CSS_SELECTOR,".bp9Aid").text

'2025.04.03'

In [ ]:
#리뷰일을 날짜형 데이터로 변경
review_date=review_date.replace('년','-').replace('월','-'),replace('일','-')
review_date=datetime.strptime(review_date,"%Y-%m-%d")
review_date

In [ ]:
#오늘로부터 1달 전 날짜 만들기
today=datetime.today()
twoyrago=today-timedelta(days=30)

In [ ]:
#리뷰들 중 가장 마지막 데이터의 날짜 추출해 날짜 데이터로 변환
review=driver.find_elements(By.CSS_SELECTOR,".RHo1pe")[-1]
end_Date=review.find_element(By.CSS_SELECTOR, "div.Jx4nYe > span.bp9Aid").text
end_date = end_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
end_date = datetime.strptime(end_date, "%Y-%m-%d")
end_date

In [ ]:
from tqdm import tqdm